# exp2 R1 2 baseline stats

**Self-contained Colab notebook.** Run the setup cells once (top to bottom), then the experiment. The four shared modules are written to `/content` automatically, so you do NOT need to upload a `shared/` folder. Edit Drive paths only in the `config_paths.py` cell below.

> Runtime: set **Runtime -> Change runtime type -> GPU** for the inference and training experiments.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
print('Drive mounted; /content on path.')

Mounted at /content/drive
Drive mounted; /content on path.


In [2]:
# scikit-image backs metrics.calc_ssim; guarantee it for every notebook
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-image'])
print('scikit-image ready')

scikit-image ready


In [3]:
%%writefile /content/config_paths.py
"""
Central path + constant configuration for all JBHI revision Colab experiments.

Derived verbatim from the author's existing notebooks:
  - COLAB_IEEE_FIGURES.ipynb      (paper baselines + EPHNOGRAM chunks + split)
  - cross_dataset_validation.ipynb (SensSmart-Tech + Trail-4 generator)

Every experiment script imports from this file so paths live in ONE place.
If your Drive layout differs, edit ONLY this file.
"""

import os
import torch

# ============================================================================
# >>> EDIT THIS BLOCK IF YOUR DRIVE MOVES <<<
# Updated 2026-07-10 for the new Google account layout. The base folder now
# sits two levels deeper under "NUST MS Research/Dataset". If everything lives
# somewhere else, change ONLY DRIVE_ROOT and OCT_DIR below.
# ============================================================================
DRIVE_ROOT = "/content/drive/MyDrive"
# Base folder that contains "Refine-Chunked", the checkpoints, etc.
OCT_DIR    = f"{DRIVE_ROOT}/NUST MS Research/Dataset/October Thesis Dataset"

# ----------------------------------------------------------------------------
# EPHNOGRAM data (training-domain test set = 453 segments, 3 subjects)
# .npz keys per chunk: 'ecg', 'pcg', 'subject_id', 'scenario'
#   ecg, pcg : float32 arrays, 64000 samples (8 s @ 8 kHz), RAW (not normalized)
#   subject_id : e.g. 'S009', 'S019', 'S023'
#   scenario   : free-text, e.g. 'rest', 'exercise bicycle stress test'
# ----------------------------------------------------------------------------
EPH_CHUNKS_DIR = f"{OCT_DIR}/Refine-Chunked"
# Subject split. keys: train/val/test_subjects. IF THIS FILE IS ABSENT the
# notebooks fall back to an arbitrary split, which gives the WRONG test set
# (the 2026-07-10 run hit this: 22 segments / 4 subjects instead of the paper's
# 453 segments / 3 subjects). Upload the real split JSON, or point this at it.
SPLIT_FILE     = f"{EPH_CHUNKS_DIR}/train_val_test_split.json"

# ----------------------------------------------------------------------------
# Model checkpoints  --  CONFIRM ALL FOUR EXIST IN THE NEW ACCOUNT
#
#   "Proposed" MUST be the Trail-4 generator (polarity fix, BatchNorm output),
#   i.e. trail4_best.pt or trail4_generator.pth. A "trial2_*" / Trail-2 file is
#   the WRONG (Tanh) model: it loads only partially under strict=False and
#   produces a near-flat, invalid reconstruction (see the 2026-07-10 run).
#
#   The three baselines lived under a SEPARATE top-level folder
#   (Paper1_ECG_Reconstruction) in the old account. Confirm they were migrated;
#   if they now sit under OCT_DIR, repoint them below.
# ----------------------------------------------------------------------------
CKPT = {
    "CNN":      f"{DRIVE_ROOT}/Paper1_ECG_Reconstruction/checkpoints/CNN_EncoderDecoder/best_model.pt",
    "UNet":     f"{DRIVE_ROOT}/Paper1_ECG_Reconstruction/checkpoints/UNet_Baseline/best_model.pt",
    "WGAN-GP":  f"{DRIVE_ROOT}/Paper1_ECG_Reconstruction/checkpoints/WGAN_GP/best_model.pt",
    "Proposed": f"{OCT_DIR}/Best-epoch-trail-4/trail4_best.pt",   # confirmed 2026-07-10
}
CKPT_PROPOSED_ALT = f"{OCT_DIR}/trail4_generator.pth"   # fallback copy (state_dict)

# ----------------------------------------------------------------------------
# SensSmart-Tech cross-dataset (32 subjects, 338 records @ 1 kHz -> 8 kHz)
#   WFDB pairs: {base}_ecg.hea/.dat and {base}_pcg.hea/.dat ; ECG lead II
# ----------------------------------------------------------------------------
SENSSMART_WFDB  = f"{OCT_DIR}/senssmarttech-database/WFDB"
SENSSMART_PREP  = f"{OCT_DIR}/milestone2_preprocessed"     # cached .npz (keys ecg/pcg)

# ----------------------------------------------------------------------------
# Output root for ALL revision artifacts (created on demand)
#   Each experiment writes to a named sub-folder here.
# ----------------------------------------------------------------------------
OUT_ROOT = f"{DRIVE_ROOT}/JBHI_Revision"

def out_dir(name):
    """Return (and create) an output sub-folder under OUT_ROOT."""
    p = os.path.join(OUT_ROOT, name)
    os.makedirs(p, exist_ok=True)
    return p

# ----------------------------------------------------------------------------
# Signal + inference constants
# ----------------------------------------------------------------------------
FS          = 8000        # sampling rate (Hz)
SEG_LEN     = 64000       # samples per segment (8 s)
NUM_SCENARIOS = 3         # rest / exercise / recovery one-hot
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# R-peak detector settings (identical to metrics.ipynb; keep fixed for reproducibility)
RPEAK_DISTANCE   = 400    # min inter-peak samples = 50 ms @ 8 kHz
RPEAK_PROMINENCE = 0.5

# SensSmart preprocessing bands (from cross_dataset_validation.ipynb)
ECG_BAND = (0.5, 40.0)
PCG_BAND = (25.0, 400.0)
FILTER_ORDER = 4

# Scenario one-hots
SCEN_REST     = [1.0, 0.0, 0.0]
SCEN_EXERCISE = [0.0, 1.0, 0.0]
SCEN_RECOVERY = [0.0, 0.0, 1.0]


def scenario_to_index(scenario_str):
    """Map EPHNOGRAM free-text scenario -> class index {0,1,2}.

    Matches the mapping used in COLAB_IEEE_FIGURES.ipynb.
    """
    s = str(scenario_str).lower()
    if "rest" in s or "laying" in s:
        return 0
    if "exercise" in s or "bicycle" in s or "treadmill" in s or "running" in s:
        return 1
    return 2

Writing /content/config_paths.py


In [4]:
%%writefile /content/models.py
"""
Model architectures for the JBHI revision experiments.

Reproduced VERBATIM from the author's COLAB_IEEE_FIGURES.ipynb (paper Table II
baselines) and models.ipynb (ablation trails), so the checkpoints in
config_paths.CKPT load cleanly.

Which class loads which checkpoint
----------------------------------
    CKPT['CNN']      -> CNNEncoderDecoder()        # no scenario, forward(x)
    CKPT['UNet']     -> UNetGenerator()            # pix2pix, forward(x)
    CKPT['WGAN-GP']  -> UNetGenerator()            # same generator as U-Net
    CKPT['Proposed'] -> ImprovedGeneratorNoTanh()  # scenario, forward(x, s_onehot)

The proposed model's forward takes a scenario one-hot; the three baselines take
only the PCG. Every inference script must branch on this (helper `run_generator`
below does it for you).

For the R1.4 component ablation, use `ConfigurableGenerator`, which exposes the
three grouped factors as switches: use_attention, use_dilated, and
output_activation in {'batchnorm','tanh','linear'}.

NOTE ON PARAM COUNTS: the submitted paper's numbers (CNN 9.8M, U-Net 13.6M,
Proposed 14.0M) do NOT match these classes (which give ~larger counts). Always
report count_params(model_loaded_from_checkpoint) computed here -- see R2.2/R2.5.
"""

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ===========================================================================
# Shared bottleneck modules
# ===========================================================================
class MultiHeadSelfAttention(nn.Module):
    """8-head self-attention over the temporal axis at the bottleneck."""
    def __init__(self, channels, num_heads=8, dropout=0.1):
        super().__init__()
        assert channels % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1, bias=False)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, C, T = x.shape
        qkv = self.qkv(x).reshape(B, 3, self.num_heads, self.head_dim, T)
        qkv = qkv.permute(1, 0, 2, 4, 3)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = self.dropout(F.softmax(attn, dim=-1))
        out = (attn @ v).transpose(2, 3).reshape(B, C, T)
        return self.proj(out) + x


class MultiScaleDilatedBlock(nn.Module):
    """Parallel dilated convs at rates 1, 2, 4 with a 1x1 fusion + residual."""
    def __init__(self, channels):
        super().__init__()
        c1 = channels // 3 + (1 if channels % 3 >= 1 else 0)
        c2 = channels // 3 + (1 if channels % 3 >= 2 else 0)
        c3 = channels // 3
        self.conv_d1 = nn.Conv1d(channels, c1, 3, padding=1, dilation=1)
        self.conv_d2 = nn.Conv1d(channels, c2, 3, padding=2, dilation=2)
        self.conv_d4 = nn.Conv1d(channels, c3, 3, padding=4, dilation=4)
        self.fusion = nn.Sequential(
            nn.Conv1d(channels, channels, 1),
            nn.BatchNorm1d(channels), nn.ReLU(inplace=True))

    def forward(self, x):
        cat = torch.cat([self.conv_d1(x), self.conv_d2(x), self.conv_d4(x)], dim=1)
        return self.fusion(cat) + x


class EnhancedResidualBlock(nn.Module):
    """Residual block with optional MHSA and MSDC (Trails 2-4)."""
    def __init__(self, channels, use_attention=False, use_dilated=True):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(channels)
        self.use_dilated = use_dilated
        if use_dilated:
            self.dilated_block = MultiScaleDilatedBlock(channels)
        self.conv2 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)
        self.use_attention = use_attention
        if use_attention:
            self.attention = MultiHeadSelfAttention(channels, num_heads=8)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        if self.use_dilated:
            out = self.dilated_block(out)
        out = self.bn2(self.conv2(out))
        if self.use_attention:
            out = self.attention(out)
        return self.relu(out + residual)


# ===========================================================================
# Paper Table II baseline: CNN Encoder-Decoder (8-level, no skips, no scenario)
# ===========================================================================
class ConvBlock(nn.Module):
    def __init__(self, i, o, k=4, s=2, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(i, o, k, s, p, bias=False),
            nn.BatchNorm1d(o), nn.LeakyReLU(0.2, inplace=True))
    def forward(self, x): return self.block(x)


class DeconvBlock(nn.Module):
    def __init__(self, i, o, k=4, s=2, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose1d(i, o, k, s, p, bias=False),
            nn.BatchNorm1d(o), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)


class CNNEncoderDecoder(nn.Module):
    """Paper CNN baseline -> loads CKPT['CNN']. forward(x)."""
    def __init__(self, in_channels=1, out_channels=1, base_filters=64):
        super().__init__()
        bf = base_filters
        self.enc1 = ConvBlock(in_channels, bf)
        self.enc2 = ConvBlock(bf, bf * 2)
        self.enc3 = ConvBlock(bf * 2, bf * 4)
        self.enc4 = ConvBlock(bf * 4, bf * 8)
        self.enc5 = ConvBlock(bf * 8, bf * 8)
        self.enc6 = ConvBlock(bf * 8, bf * 8)
        self.enc7 = ConvBlock(bf * 8, bf * 8)
        self.enc8 = ConvBlock(bf * 8, bf * 8)
        self.dec1 = DeconvBlock(bf * 8, bf * 8)
        self.dec2 = DeconvBlock(bf * 8, bf * 8)
        self.dec3 = DeconvBlock(bf * 8, bf * 8)
        self.dec4 = DeconvBlock(bf * 8, bf * 8)
        self.dec5 = DeconvBlock(bf * 8, bf * 4)
        self.dec6 = DeconvBlock(bf * 4, bf * 2)
        self.dec7 = DeconvBlock(bf * 2, bf)
        self.final = nn.ConvTranspose1d(bf, out_channels, 4, 2, 1)

    def forward(self, x):
        n = x.size(2)
        e = self.enc1(x); e = self.enc2(e); e = self.enc3(e); e = self.enc4(e)
        e = self.enc5(e); e = self.enc6(e); e = self.enc7(e); e = self.enc8(e)
        d = self.dec1(e); d = self.dec2(d); d = self.dec3(d); d = self.dec4(d)
        d = self.dec5(d); d = self.dec6(d); d = self.dec7(d)
        out = self.final(d)
        if out.size(2) != n:
            diff = n - out.size(2)
            out = F.pad(out, (diff // 2, diff - diff // 2))
        return out


# ===========================================================================
# Paper Table II baseline: pix2pix U-Net (shared by U-Net and WGAN-GP)
# ===========================================================================
class UNetBlock(nn.Module):
    def __init__(self, i, o, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv1d(i, o, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.BatchNorm1d(o))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)


class UNetUpBlock(nn.Module):
    def __init__(self, i, o, dropout=0.0):
        super().__init__()
        layers = [nn.ConvTranspose1d(i, o, 4, 2, 1, bias=False),
                  nn.BatchNorm1d(o), nn.ReLU(inplace=True)]
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x, skip):
        x = self.block(x)
        if x.size(2) != skip.size(2):
            diff = skip.size(2) - x.size(2)
            x = F.pad(x, (diff // 2, diff - diff // 2))
        return torch.cat([x, skip], dim=1)


class UNetGenerator(nn.Module):
    """Paper U-Net / WGAN-GP generator -> loads CKPT['UNet'] or CKPT['WGAN-GP']. forward(x)."""
    def __init__(self, in_channels=1, out_channels=1, features=64):
        super().__init__()
        f = features
        self.e1 = UNetBlock(in_channels, f, normalize=False)
        self.e2 = UNetBlock(f, f * 2); self.e3 = UNetBlock(f * 2, f * 4)
        self.e4 = UNetBlock(f * 4, f * 8); self.e5 = UNetBlock(f * 8, f * 8)
        self.e6 = UNetBlock(f * 8, f * 8); self.e7 = UNetBlock(f * 8, f * 8)
        self.e8 = UNetBlock(f * 8, f * 8, normalize=False)
        self.d1 = UNetUpBlock(f * 8, f * 8, dropout=0.5)
        self.d2 = UNetUpBlock(f * 16, f * 8, dropout=0.5)
        self.d3 = UNetUpBlock(f * 16, f * 8, dropout=0.5)
        self.d4 = UNetUpBlock(f * 16, f * 8)
        self.d5 = UNetUpBlock(f * 16, f * 4); self.d6 = UNetUpBlock(f * 8, f * 2)
        self.d7 = UNetUpBlock(f * 4, f)
        self.final = nn.ConvTranspose1d(f * 2, out_channels, 4, 2, 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(e1); e3 = self.e3(e2); e4 = self.e4(e3)
        e5 = self.e5(e4); e6 = self.e6(e5); e7 = self.e7(e6); e8 = self.e8(e7)
        d = self.d1(e8, e7); d = self.d2(d, e6); d = self.d3(d, e5); d = self.d4(d, e4)
        d = self.d5(d, e3); d = self.d6(d, e2); d = self.d7(d, e1)
        out = self.final(d)
        if out.size(2) != x.size(2):
            diff = x.size(2) - out.size(2)
            out = F.pad(out, (diff // 2, diff - diff // 2))
        return out


# ===========================================================================
# Proposed generator (Trail-4): U-Net + MHSA + MSDC + scenario, BatchNorm output
# ===========================================================================
class ImprovedGeneratorNoTanh(nn.Module):
    """Proposed model -> loads CKPT['Proposed']. forward(x, scenario_onehot)."""
    def __init__(self, signal_length=64000, base_channels=64, num_scenarios=3):
        super().__init__()
        bc = base_channels
        self.scenario_embed = nn.Linear(num_scenarios, bc)
        self.enc1 = nn.Sequential(nn.Conv1d(1, bc, 7, 2, 3), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv1d(bc, bc*2, 5, 2, 2), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv1d(bc*2, bc*4, 5, 2, 2), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.enc4 = nn.Sequential(nn.Conv1d(bc*4, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.enc5 = nn.Sequential(nn.Conv1d(bc*8, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.bottleneck = nn.Sequential(
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True))
        self.dec5 = nn.Sequential(nn.ConvTranspose1d(bc*8, bc*8, 4, 2, 1), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.dec4 = nn.Sequential(nn.Conv1d(bc*16, bc*8, 1), nn.ConvTranspose1d(bc*8, bc*4, 4, 2, 1), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.dec3 = nn.Sequential(nn.Conv1d(bc*8, bc*4, 1), nn.ConvTranspose1d(bc*4, bc*2, 4, 2, 1), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.Conv1d(bc*4, bc*2, 1), nn.ConvTranspose1d(bc*2, bc, 4, 2, 1), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.dec1 = nn.Sequential(nn.Conv1d(bc*2, bc, 1), nn.ConvTranspose1d(bc, 1, 4, 2, 1), nn.BatchNorm1d(1))

    def forward(self, x, scenario_onehot):
        s = self.scenario_embed(scenario_onehot).unsqueeze(2)
        e1 = self.enc1(x) + s
        e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3); e5 = self.enc5(e4)
        b = self.bottleneck(e5)
        d5 = torch.cat([self.dec5(b), e4], dim=1)
        d4 = torch.cat([self.dec4(d5), e3], dim=1)
        d3 = torch.cat([self.dec3(d4), e2], dim=1)
        d2 = torch.cat([self.dec2(d3), e1], dim=1)
        return self.dec1(d2)


# ===========================================================================
# Configurable generator for the R1.4 component ablation
# ===========================================================================
class ConfigurableGenerator(nn.Module):
    """Ablation generator: toggle each grouped factor independently.

    use_attention : add MHSA in the 4 bottleneck blocks
    use_dilated   : add MSDC in the 4 bottleneck blocks
    output_activation : 'batchnorm' (proposed) | 'tanh' | 'linear'
    use_scenario  : condition on the scenario one-hot (keep True to match proposed)

    Ablation rows the reviewer asked for map to:
        U-Net only          -> use_attention=False, use_dilated=False
        U-Net + MSDC        -> use_attention=False, use_dilated=True
        U-Net + MHSA        -> use_attention=True,  use_dilated=False
        U-Net + both        -> use_attention=True,  use_dilated=True
        Proposed + Tanh     -> both True, output_activation='tanh'
        Proposed + linear   -> both True, output_activation='linear'
        Proposed (BatchNorm)-> both True, output_activation='batchnorm'
    Polarity-on/off is a DATA-loader switch; adversarial-on/off is a TRAINING
    switch -- neither is an architecture change, so control them in the trainer.
    """
    def __init__(self, base_channels=64, num_scenarios=3,
                 use_attention=True, use_dilated=True,
                 output_activation="batchnorm", use_scenario=True):
        super().__init__()
        bc = base_channels
        self.use_scenario = use_scenario
        if use_scenario:
            self.scenario_embed = nn.Linear(num_scenarios, bc)
        self.enc1 = nn.Sequential(nn.Conv1d(1, bc, 7, 2, 3), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv1d(bc, bc*2, 5, 2, 2), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv1d(bc*2, bc*4, 5, 2, 2), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.enc4 = nn.Sequential(nn.Conv1d(bc*4, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.enc5 = nn.Sequential(nn.Conv1d(bc*8, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.bottleneck = nn.Sequential(*[
            EnhancedResidualBlock(bc*8, use_attention=use_attention, use_dilated=use_dilated)
            for _ in range(4)])
        self.dec5 = nn.Sequential(nn.ConvTranspose1d(bc*8, bc*8, 4, 2, 1), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.dec4 = nn.Sequential(nn.Conv1d(bc*16, bc*8, 1), nn.ConvTranspose1d(bc*8, bc*4, 4, 2, 1), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.dec3 = nn.Sequential(nn.Conv1d(bc*8, bc*4, 1), nn.ConvTranspose1d(bc*4, bc*2, 4, 2, 1), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.Conv1d(bc*4, bc*2, 1), nn.ConvTranspose1d(bc*2, bc, 4, 2, 1), nn.BatchNorm1d(bc), nn.ReLU(True))
        head = [nn.Conv1d(bc*2, bc, 1), nn.ConvTranspose1d(bc, 1, 4, 2, 1)]
        if output_activation == "batchnorm":
            head.append(nn.BatchNorm1d(1))
        elif output_activation == "tanh":
            head.append(nn.Tanh())
        elif output_activation == "linear":
            pass
        else:
            raise ValueError(f"unknown output_activation {output_activation}")
        self.dec1 = nn.Sequential(*head)

    def forward(self, x, scenario_onehot=None):
        e1 = self.enc1(x)
        if self.use_scenario and scenario_onehot is not None:
            e1 = e1 + self.scenario_embed(scenario_onehot).unsqueeze(2)
        e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3); e5 = self.enc5(e4)
        b = self.bottleneck(e5)
        d5 = torch.cat([self.dec5(b), e4], dim=1)
        d4 = torch.cat([self.dec4(d5), e3], dim=1)
        d3 = torch.cat([self.dec3(d4), e2], dim=1)
        d2 = torch.cat([self.dec2(d3), e1], dim=1)
        return self.dec1(d2)


# ===========================================================================
# Discriminator (for R1.4 adversarial retraining)
# ===========================================================================
class SpectralNormDiscriminator(nn.Module):
    """Spectrally-normalized PatchGAN critic with scenario conditioning."""
    def __init__(self, base_channels=64, num_scenarios=3):
        super().__init__()
        bc = base_channels
        self.scenario_embed = nn.Linear(num_scenarios, bc)
        sn = nn.utils.spectral_norm
        self.conv1 = sn(nn.Conv1d(2, bc, 7, 2, 3))
        self.conv2 = sn(nn.Conv1d(bc, bc*2, 5, 2, 2)); self.bn2 = nn.BatchNorm1d(bc*2)
        self.conv3 = sn(nn.Conv1d(bc*2, bc*4, 5, 2, 2)); self.bn3 = nn.BatchNorm1d(bc*4)
        self.conv4 = sn(nn.Conv1d(bc*4, bc*8, 5, 2, 2)); self.bn4 = nn.BatchNorm1d(bc*8)
        self.conv5 = sn(nn.Conv1d(bc*8, bc*8, 5, 2, 2)); self.bn5 = nn.BatchNorm1d(bc*8)
        self.pool = nn.AdaptiveAvgPool1d(1); self.fc = nn.Linear(bc*8, 1)
        self.lrelu = nn.LeakyReLU(0.2, True)

    def forward(self, pcg, ecg, scenario_onehot):
        x = torch.cat([pcg, ecg], dim=1)
        s = self.scenario_embed(scenario_onehot).unsqueeze(2)
        x = self.lrelu(self.conv1(x)) + s
        x = self.lrelu(self.bn2(self.conv2(x)))
        x = self.lrelu(self.bn3(self.conv3(x)))
        x = self.lrelu(self.bn4(self.conv4(x)))
        x = self.lrelu(self.bn5(self.conv5(x)))
        return torch.sigmoid(self.fc(self.pool(x).squeeze(2)))


# ===========================================================================
# Loading + utility helpers
# ===========================================================================
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def load_model(model, checkpoint_path, device="cpu", strict=False):
    """Load a checkpoint into `model`, handling common save formats."""
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict):
        for key in ("generator_state_dict", "model_state_dict", "state_dict"):
            if key in ckpt:
                ckpt = ckpt[key]; break
    state = {k.replace("module.", ""): v for k, v in ckpt.items()} \
            if isinstance(ckpt, dict) else ckpt
    result = model.load_state_dict(state, strict=strict)
    # Loud guard: a large key mismatch means the checkpoint does not fit this
    # architecture (e.g. a Trail-2/trial2 file loaded into the Trail-4 model).
    n_model = max(len(model.state_dict()), 1)
    n_missing, n_unexpected = len(result.missing_keys), len(result.unexpected_keys)
    tag = os.path.basename(checkpoint_path)
    if n_missing or n_unexpected:
        flag = "  <-- LIKELY WRONG CHECKPOINT FOR THIS MODEL" if n_missing > 0.10 * n_model else ""
        print(f"[load_model] {tag}: {n_missing}/{n_model} params missing, "
              f"{n_unexpected} unexpected{flag}")
    else:
        print(f"[load_model] {tag}: exact match ({n_model} tensors).")
    model.to(device).eval()
    return model, result


def build_paper_models(ckpt_dict, device="cpu"):
    """Load all four paper models. Returns {name: model}."""
    models = {}
    models["CNN"], _     = load_model(CNNEncoderDecoder(), ckpt_dict["CNN"], device)
    models["UNet"], _    = load_model(UNetGenerator(), ckpt_dict["UNet"], device)
    models["WGAN-GP"], _ = load_model(UNetGenerator(), ckpt_dict["WGAN-GP"], device)
    models["Proposed"], _= load_model(ImprovedGeneratorNoTanh(), ckpt_dict["Proposed"], device)
    return models


def run_generator(model, pcg_tensor, scenario_tensor, is_proposed):
    """Uniform inference call. `is_proposed` selects the (x, s) signature."""
    with torch.no_grad():
        if is_proposed:
            out = model(pcg_tensor, scenario_tensor)
        else:
            out = model(pcg_tensor)
    return out.squeeze().cpu().numpy()

Writing /content/models.py


In [5]:
%%writefile /content/metrics.py
"""
Metrics for the JBHI revision.

Two families:
  (A) Signal-fidelity metrics  -- identical to the submitted paper's metrics.ipynb
                                  (PSNR, SNR, SSIM, Pearson r, MAE, RMSE,
                                   R-peak AMPLITUDE error, QRS-width error).
  (B) Temporal R-peak DETECTION metrics  -- NEW, added to answer Reviewer 1 #1.
       These detect R-peaks INDEPENDENTLY in the reconstructed ECG and match
       them to ground-truth peaks within a tolerance window, then report
       sensitivity, PPV, F1, FP, FN, timing-error stats, RR-interval error,
       and heart-rate error. This is the metric the paper was missing.

Keep the detector settings fixed (config_paths.RPEAK_*) so every experiment and
the paper agree.
"""

import numpy as np
import scipy.signal as signal
from scipy.stats import pearsonr


# ---------------------------------------------------------------------------
# R-peak detection (single, shared definition)
# ---------------------------------------------------------------------------
def detect_r_peaks(ecg, fs=8000, distance=400, prominence=0.5):
    """scipy find_peaks on an ECG trace. Returns integer sample indices."""
    peaks, _ = signal.find_peaks(np.asarray(ecg).ravel(),
                                 distance=distance, prominence=prominence)
    return peaks


# ---------------------------------------------------------------------------
# (A) Signal-fidelity metrics
# ---------------------------------------------------------------------------
def calc_psnr(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    mse = np.mean((real - fake) ** 2)
    if mse < 1e-10:
        return 100.0
    peak = np.max(np.abs(real))
    return float(20 * np.log10(peak / np.sqrt(mse)))


def calc_snr(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    noise = real - fake
    return float(10 * np.log10(np.sum(real ** 2) / (np.sum(noise ** 2) + 1e-12)))


def calc_correlation(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    try:
        c, _ = pearsonr(real, fake)
        return float(c)
    except Exception:
        return 0.0


def calc_ssim(real, fake):
    from skimage.metrics import structural_similarity as compare_ssim  # lazy: Colab-only dep
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    r = (real - real.min()) / (real.max() - real.min() + 1e-8)
    f = (fake - fake.min()) / (fake.max() - fake.min() + 1e-8)
    return float(compare_ssim(r, f, data_range=1.0))


def calc_mae(real, fake):
    return float(np.mean(np.abs(np.asarray(real).ravel() - np.asarray(fake).ravel())))


def calc_rmse(real, fake):
    d = np.asarray(real).ravel() - np.asarray(fake).ravel()
    return float(np.sqrt(np.mean(d ** 2)))


def calc_r_peak_amplitude_error(real, fake, fs=8000, **kw):
    """Paper's 'R-peak error' -> renamed 'R-peak AMPLITUDE error'.

    Detects R-peaks on the GROUND TRUTH only and averages |y(t_k)-yhat(t_k)|.
    Kept ONLY as a fidelity measure; it is NOT a timing metric (Reviewer 1 #1).
    Returns np.nan when the ground truth has no detectable R-peaks (report the
    valid-N denominator alongside the mean).
    """
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    peaks = detect_r_peaks(real, fs=fs, **kw)
    if len(peaks) == 0:
        return np.nan
    return float(np.mean(np.abs(real[peaks] - fake[peaks])))


def _qrs_width_at_peak(ecg, peak_idx, fs=8000, threshold=0.5, search_ms=25):
    amp = abs(ecg[peak_idx]); thr = threshold * amp
    win = int(search_ms / 1000 * fs)
    q = peak_idx
    for j in range(peak_idx - 1, max(peak_idx - win, 0), -1):
        if abs(ecg[j]) < thr:
            q = j; break
    s = peak_idx
    for j in range(peak_idx + 1, min(peak_idx + win, len(ecg))):
        if abs(ecg[j]) < thr:
            s = j; break
    return (s - q) / fs * 1000.0


def calc_qrs_width_error(real, fake, fs=8000, **kw):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    peaks = detect_r_peaks(real, fs=fs, **kw)
    if len(peaks) == 0:
        return np.nan
    errs = [abs(_qrs_width_at_peak(real, p, fs) - _qrs_width_at_peak(fake, p, fs))
            for p in peaks]
    return float(np.mean(errs))


# ---------------------------------------------------------------------------
# (B) Temporal R-peak DETECTION metrics  (NEW -- Reviewer 1 #1)
# ---------------------------------------------------------------------------
def match_peaks(true_peaks, pred_peaks, tol_samples):
    """Greedy nearest-neighbour matching within a tolerance window.

    Each true peak claims the closest unused predicted peak within tol_samples.
    Returns (matches, TP, FP, FN, signed_timing_errors_samples).
    signed error = pred_index - true_index (positive = reconstructed peak late).
    """
    true_peaks = list(np.asarray(true_peaks).astype(int))
    pred_peaks = list(np.asarray(pred_peaks).astype(int))
    used, matches, terr = set(), [], []
    for tp in true_peaks:
        cands = [(abs(pp - tp), pp) for pp in pred_peaks
                 if pp not in used and abs(pp - tp) <= tol_samples]
        if cands:
            _, pp = min(cands)
            used.add(pp); matches.append((tp, pp)); terr.append(pp - tp)
    TP = len(matches)
    FN = len(true_peaks) - TP
    FP = len(pred_peaks) - TP
    return matches, TP, FP, FN, np.asarray(terr, dtype=float)


def temporal_rpeak_metrics(true_ecg, pred_ecg, fs=8000, tol_ms=50,
                           distance=400, prominence=0.5):
    """Full temporal detection report for ONE segment.

    Returns a dict with detection (Se/PPV/F1/FP/FN), timing-error (ms),
    RR-interval error (ms), and heart-rate error (bpm). Segments with no
    ground-truth peaks return n_true=0 and NaN rates (exclude from means, but
    COUNT them -- a reconstruction that yields peaks where truth has none is a
    real failure; see caller handling).
    """
    tol = int(round(tol_ms / 1000 * fs))
    t_pk = detect_r_peaks(true_ecg, fs, distance, prominence)
    p_pk = detect_r_peaks(pred_ecg, fs, distance, prominence)

    matches, TP, FP, FN, terr = match_peaks(t_pk, p_pk, tol)
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    ppv  = TP / (TP + FP) if (TP + FP) else np.nan
    f1   = (2 * sens * ppv / (sens + ppv)
            if (sens and ppv and not np.isnan(sens) and not np.isnan(ppv)
                and (sens + ppv) > 0) else np.nan)

    terr_ms = np.abs(terr) / fs * 1000.0
    mae_t   = float(np.mean(terr_ms)) if terr_ms.size else np.nan
    med_t   = float(np.median(terr_ms)) if terr_ms.size else np.nan
    iqr_t   = float(np.percentile(terr_ms, 75) - np.percentile(terr_ms, 25)) \
              if terr_ms.size else np.nan

    # RR-interval error on consecutive MATCHED pairs only (avoids missed-beat bias)
    rr_err = np.nan
    if len(matches) >= 2:
        tp_sorted = sorted(matches)
        rr_true, rr_pred = [], []
        for (t0, p0), (t1, p1) in zip(tp_sorted[:-1], tp_sorted[1:]):
            rr_true.append((t1 - t0) / fs * 1000.0)
            rr_pred.append((p1 - p0) / fs * 1000.0)
        rr_err = float(np.mean(np.abs(np.array(rr_true) - np.array(rr_pred))))

    # Heart-rate error (bpm), estimator = 60000 / median-RR
    def _hr(pk):
        if len(pk) < 2:
            return np.nan
        return 60000.0 / np.median(np.diff(pk) / fs * 1000.0)
    hr_true, hr_pred = _hr(t_pk), _hr(p_pk)
    hr_err = float(abs(hr_true - hr_pred)) if not (np.isnan(hr_true) or np.isnan(hr_pred)) else np.nan

    return {
        "n_true": int(len(t_pk)), "n_pred": int(len(p_pk)),
        "TP": TP, "FP": FP, "FN": FN,
        "sensitivity": sens, "ppv": ppv, "f1": f1,
        "mae_timing_ms": mae_t, "median_timing_ms": med_t, "iqr_timing_ms": iqr_t,
        "rr_error_ms": rr_err,
        "hr_true_bpm": hr_true, "hr_pred_bpm": hr_pred, "hr_error_bpm": hr_err,
    }


def aggregate_temporal(rows):
    """Aggregate a list of per-segment temporal_rpeak_metrics dicts.

    Detection totals (TP/FP/FN) are POOLED then Se/PPV/F1 recomputed (the
    correct way to aggregate detection over segments). Timing/RR/HR errors are
    averaged over segments where they are defined; the valid-N is reported so
    the denominator is transparent.
    """
    rows = [r for r in rows if r is not None]
    TP = sum(r["TP"] for r in rows)
    FP = sum(r["FP"] for r in rows)
    FN = sum(r["FN"] for r in rows)
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    ppv  = TP / (TP + FP) if (TP + FP) else np.nan
    f1   = 2 * sens * ppv / (sens + ppv) if (sens + ppv) else np.nan

    def _mean(key):
        v = [r[key] for r in rows if r[key] is not None and not np.isnan(r[key])]
        return (float(np.mean(v)), float(np.std(v)), len(v)) if v else (np.nan, np.nan, 0)

    out = {"n_segments": len(rows), "TP": TP, "FP": FP, "FN": FN,
           "sensitivity": sens, "ppv": ppv, "f1": f1}
    for k in ["mae_timing_ms", "median_timing_ms", "iqr_timing_ms",
              "rr_error_ms", "hr_error_bpm"]:
        m, s, n = _mean(k)
        out[k + "_mean"], out[k + "_std"], out[k + "_validN"] = m, s, n
    return out


# ---------------------------------------------------------------------------
# Preprocessing helpers (shared with the paper pipeline)
# ---------------------------------------------------------------------------
def ensure_positive_polarity(ecg):
    """Flip so R-peaks point up. Applied to the ECG TARGET only (never PCG)."""
    ecg = np.asarray(ecg)
    return -ecg if np.abs(ecg.min()) > np.abs(ecg.max()) else ecg


def normalize_zscore(sig):
    sig = np.asarray(sig, dtype=np.float32)
    return ((sig - sig.mean()) / (sig.std() + 1e-8)).astype(np.float32)

Writing /content/metrics.py


In [6]:
%%writefile /content/ieee_style.py
"""
IEEE double-column figure style for the JBHI revision.

Design goals (per IEEE Transactions / J-BHI figure rules):
  * Vector output (PDF) + high-DPI PNG (600) for every figure.
  * Serif face (Times) to match IEEE body text; DejaVu Serif fallback on Colab.
  * BOLD axis labels that always carry explicit units.
  * Thick spines and ticks so lines stay legible at column scale.
  * constrained_layout everywhere -> no text/label/legend overlaps.
  * Column widths fixed to IEEE spec: single = 3.5 in, double = 7.16 in.

Usage
-----
    from ieee_style import set_ieee_style, new_fig, finalize_axes, save_fig
    set_ieee_style()
    fig, ax = new_fig("single", height=2.2)
    ax.plot(t, y)
    finalize_axes(ax, xlabel="Time (s)", ylabel="Amplitude (a.u.)")
    save_fig(fig, out_dir, "fig_name")   # writes fig_name.pdf AND fig_name.png
"""

import matplotlib
matplotlib.use("Agg")            # headless-safe; remove if you want inline display
import matplotlib.pyplot as plt

# IEEE column widths (inches)
COL_SINGLE = 3.5
COL_DOUBLE = 7.16

# Colour-blind-safe palette (Okabe-Ito), stable role -> colour mapping
COLORS = {
    "truth":    "#009E73",   # bluish green  -> ground truth
    "proposed": "#0072B2",   # blue          -> proposed model
    "cnn":      "#D55E00",   # vermillion    -> CNN
    "unet":     "#CC79A7",   # reddish purple-> U-Net
    "wgan":     "#E69F00",   # orange        -> WGAN-GP
    "pcg":      "#666666",   # grey          -> PCG input
    "accent":   "#0072B2",
    "muted":    "#999999",
}


def set_ieee_style():
    """Apply the global rcParams. Call once at the top of every script."""
    plt.rcParams.update({
        "figure.dpi":        150,
        "savefig.dpi":       600,
        "savefig.bbox":      "tight",
        "savefig.pad_inches": 0.02,
        "figure.constrained_layout.use": True,
        "font.size":         8,
        "font.family":       "serif",
        "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset":  "dejavuserif",
        "text.usetex":       False,
        "axes.labelsize":    9,
        "axes.labelweight":  "bold",
        "axes.titlesize":    9,
        "axes.titleweight":  "bold",
        "axes.linewidth":    1.1,
        "axes.grid":         True,
        "grid.alpha":        0.25,
        "grid.linewidth":    0.4,
        "xtick.labelsize":   8,
        "ytick.labelsize":   8,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.major.size":  3.5,
        "ytick.major.size":  3.5,
        "xtick.direction":   "out",
        "ytick.direction":   "out",
        "legend.fontsize":   7.5,
        "legend.framealpha": 0.92,
        "legend.edgecolor":  "0.7",
        "legend.handlelength": 1.6,
        "lines.linewidth":   1.0,
        "lines.solid_capstyle": "round",
    })


def new_fig(width="single", height=2.2, nrows=1, ncols=1, **kw):
    """Create a figure sized to an IEEE column.

    width : 'single' (3.5 in), 'double' (7.16 in), or a float in inches.
    height: figure height in inches.
    Returns (fig, ax) or (fig, axes) exactly like plt.subplots.
    """
    w = {"single": COL_SINGLE, "double": COL_DOUBLE}.get(width, width)
    fig, ax = plt.subplots(nrows, ncols, figsize=(w, height), **kw)
    return fig, ax


def finalize_axes(ax, xlabel=None, ylabel=None, title=None,
                  legend=False, legend_loc="best", legend_ncol=1):
    """Apply bold labels with units, thicken spines, tidy ticks.

    Always pass xlabel/ylabel WITH units, e.g. 'Time (ms)', 'PSNR (dB)'.
    """
    if xlabel:
        ax.set_xlabel(xlabel, fontweight="bold")
    if ylabel:
        ax.set_ylabel(ylabel, fontweight="bold")
    if title:
        ax.set_title(title, fontweight="bold")
    for spine in ax.spines.values():
        spine.set_linewidth(1.1)
    ax.tick_params(axis="both", which="major", width=1.0, labelsize=8)
    if legend:
        leg = ax.legend(loc=legend_loc, ncol=legend_ncol, frameon=True)
        leg.get_frame().set_linewidth(0.6)
    return ax


def save_fig(fig, out_directory, name):
    """Save both a vector PDF (for LaTeX) and a 600-dpi PNG (for preview)."""
    import os
    pdf = os.path.join(out_directory, f"{name}.pdf")
    png = os.path.join(out_directory, f"{name}.png")
    fig.savefig(pdf)
    fig.savefig(png)
    plt.close(fig)
    print(f"  saved  {pdf}")
    print(f"  saved  {png}")
    return pdf, png

Writing /content/ieee_style.py


In [7]:
import importlib
import config_paths, models, metrics, ieee_style
for m in (config_paths, models, metrics, ieee_style):
    importlib.reload(m)
ieee_style.set_ieee_style()
print('shared modules ready | device =', config_paths.DEVICE)

shared modules ready | device = cuda


## Experiment

# exp2 - Per-segment baseline metrics + paired statistics (Reviewer 1, comment 2)

**Goal.** Reviewer 1 asks for a statistically defensible comparison of the
Proposed model against the three paper baselines, and for honest uncertainty
on the headline correlation given that the EPHNOGRAM test set is only three
continuous single-session recordings (effective N ~ 3).

This script:
1. loads all four paper models (CNN, U-Net, WGAN-GP, Proposed);
2. runs each over every one of the 453 EPHNOGRAM test segments and computes
   per-segment PSNR, Pearson correlation, SSIM, RMSE, and R-peak amplitude
   error; writes one tidy long CSV (one row per segment x model);
3. runs paired Wilcoxon signed-rank tests (Proposed vs each baseline, per
   metric) with Holm-Bonferroni correction across the whole family;
4. computes a subject-clustered bootstrap 95% CI for the Proposed correlation
   (resampling the three recordings) alongside the naive segment-level CI;
5. writes per-subject and per-scenario summary tables;
6. emits JSON, paste-ready LaTeX tables with significance markers, and two
   IEEE double-column figures.

**Pipeline note (must match the paper).** Every ECG target is polarity-fixed
then z-scored; the PCG input is z-scored; all fidelity metrics are computed on
the z-scored ECG versus the raw generator output. No target-std rescaling is
applied here (that rescaling is the separate R1.8 confound). This exact
pipeline reproduces the verified per-subject numbers (e.g. S023 R-peak
amplitude error 0.067).

All paths come from `config_paths`. Outputs land in
`config_paths.out_dir('exp2_baseline_stats')`. The script is resumable: the
per-segment CSV is cached and reused, and the inference loop skips segments
already present in the cache.

In [8]:
# ---------------------------------------------------------------------------
# Cell 1 - Environment setup (Colab)
# ---------------------------------------------------------------------------
# On Colab, mount Drive and put the shared/ modules on the path. You MUST copy
# the local shared/ folder to Drive first (or paste the four modules into a cell):
#     E:\...\Revision\Colab_Experiments\shared\  ->  MyDrive/JBHI_Revision/shared/
#
# If you are running this OUTSIDE Colab, comment out the drive.mount line and set
# SHARED_DIR to the local shared/ path.
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("[note] google.colab not available; assuming a local run.")

import sys, os

SHARED_DIR = '/content/drive/MyDrive/JBHI_Revision/shared'
if not os.path.isdir(SHARED_DIR):
    # Local fallback: shared/ sits next to this script.
    _local = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'shared')
    SHARED_DIR = _local if os.path.isdir(_local) else SHARED_DIR
sys.path.append(SHARED_DIR)
print(f"[note] importing shared modules from: {SHARED_DIR}")
print("[note] If import fails, copy the local shared/ folder to Drive OR paste "
      "config_paths.py, models.py, metrics.py, ieee_style.py into a cell.")

if IN_COLAB:
    # scikit-image supplies SSIM; the rest ship with Colab.
    os.system('pip install -q scikit-image')

import config_paths
import models as M
import metrics as MET
import ieee_style as IE

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[note] importing shared modules from: /content/drive/MyDrive/JBHI_Revision/shared
[note] If import fails, copy the local shared/ folder to Drive OR paste config_paths.py, models.py, metrics.py, ieee_style.py into a cell.


In [9]:
# ---------------------------------------------------------------------------
# Cell 2 - Config, constants, output folder, reproducibility
# ---------------------------------------------------------------------------
import json
import glob
import numpy as np
import pandas as pd
import torch

DEVICE = config_paths.DEVICE
OUT = config_paths.out_dir('exp2_baseline_stats')
print(f"[note] device: {DEVICE}")
print(f"[note] output folder: {OUT}")

# Set True to force a full recompute of the per-segment metrics (ignore cache).
FORCE_RECOMPUTE = False

# Bootstrap settings (fixed seed => reproducible CIs).
N_BOOT = 10000
RNG = np.random.default_rng(2026)

# Metric registry: key, display label (with unit), direction ('max' better or
# 'min' better), and print precision. RMSE and R-peak amplitude error are on the
# z-scored ECG, so their unit is arbitrary units (a.u.).
METRICS = [
    ("psnr",          "PSNR (dB)",              "max", 2),
    ("corr",          "Corr. r",                "max", 3),
    ("ssim",          "SSIM",                   "max", 3),
    ("rmse",          "RMSE (a.u.)",            "min", 3),
    ("rpeak_amp_err", "R-peak amp. err. (a.u.)","min", 3),
]
METRIC_KEYS = [m[0] for m in METRICS]

MODEL_ORDER = ["CNN", "UNet", "WGAN-GP", "Proposed"]
BASELINES = ["CNN", "UNet", "WGAN-GP"]

# Colour + pretty-name maps (colour-blind-safe roles from ieee_style).
MODEL_COLOR = {
    "CNN":      IE.COLORS["cnn"],
    "UNet":     IE.COLORS["unet"],
    "WGAN-GP":  IE.COLORS["wgan"],
    "Proposed": IE.COLORS["proposed"],
}
MODEL_PRETTY = {"CNN": "CNN", "UNet": "U-Net", "WGAN-GP": "WGAN-GP", "Proposed": "Proposed"}
SCEN_NAME = {0: "rest", 1: "exercise", 2: "recovery"}

[note] device: cuda
[note] output folder: /content/drive/MyDrive/JBHI_Revision/exp2_baseline_stats


In [10]:
# ---------------------------------------------------------------------------
# Cell 3 - Load the four paper models (guarded / resumable)
# ---------------------------------------------------------------------------
# Preferred path: models.build_paper_models loads all four at once. If any
# checkpoint is missing we fall back to loading each model individually so a
# single missing baseline does not abort the whole run.
def load_models_guarded():
    loaded = {}
    try:
        loaded = M.build_paper_models(config_paths.CKPT, device=DEVICE)
        print("[ok] build_paper_models loaded all four checkpoints.")
        return loaded
    except Exception as e:
        print(f"[warn] build_paper_models failed ({e}); loading models individually.")

    specs = [
        ("CNN",      M.CNNEncoderDecoder,      config_paths.CKPT["CNN"]),
        ("UNet",     M.UNetGenerator,          config_paths.CKPT["UNet"]),
        ("WGAN-GP",  M.UNetGenerator,          config_paths.CKPT["WGAN-GP"]),
        ("Proposed", M.ImprovedGeneratorNoTanh, config_paths.CKPT["Proposed"]),
    ]
    for name, cls, path in specs:
        if not os.path.exists(path):
            print(f"[warn] checkpoint missing for {name}: {path} -> skipping model.")
            continue
        try:
            mdl, missing = M.load_model(cls(), path, device=DEVICE)
            loaded[name] = mdl
            print(f"[ok] loaded {name} from {path}")
        except Exception as ee:
            print(f"[warn] could not load {name}: {ee}")
    return loaded

MODELS = load_models_guarded()
AVAILABLE = [m for m in MODEL_ORDER if m in MODELS]
if not AVAILABLE:
    raise SystemExit("[fatal] no models could be loaded; check config_paths.CKPT.")
print(f"[note] models available: {AVAILABLE}")

# Report verified parameter counts for the loaded models (feeds R2.2/R2.5).
for name in AVAILABLE:
    print(f"       params[{name}] = {M.count_params(MODELS[name]):,}")

[warn] build_paper_models failed ([Errno 2] No such file or directory: '/content/drive/MyDrive/Paper1_ECG_Reconstruction/checkpoints/CNN_EncoderDecoder/best_model.pt'); loading models individually.
[warn] checkpoint missing for CNN: /content/drive/MyDrive/Paper1_ECG_Reconstruction/checkpoints/CNN_EncoderDecoder/best_model.pt -> skipping model.
[warn] checkpoint missing for UNet: /content/drive/MyDrive/Paper1_ECG_Reconstruction/checkpoints/UNet_Baseline/best_model.pt -> skipping model.
[warn] checkpoint missing for WGAN-GP: /content/drive/MyDrive/Paper1_ECG_Reconstruction/checkpoints/WGAN_GP/best_model.pt -> skipping model.
[load_model] trail4_best.pt: exact match (200 tensors).
[ok] loaded Proposed from /content/drive/MyDrive/NUST MS Research/Dataset/October Thesis Dataset/Best-epoch-trail-4/trail4_best.pt
[note] models available: ['Proposed']
       params[Proposed] = 19,315,715


In [11]:
# ---------------------------------------------------------------------------
# Cell 4 - Build the EPHNOGRAM test-segment index
# ---------------------------------------------------------------------------
# Each chunk .npz holds ONE segment (ecg, pcg = 64000 samples, plus subject_id
# and scenario). The test set is every chunk whose subject_id is a test subject.
def build_test_index():
    if not os.path.exists(config_paths.SPLIT_FILE):
        print(f"[warn] split file missing: {config_paths.SPLIT_FILE}")
        return []
    with open(config_paths.SPLIT_FILE, "r") as f:
        splits = json.load(f)
    test_subjects = set(splits["test_subjects"])
    print(f"[note] test subjects from split: {sorted(test_subjects)}")

    all_chunks = sorted(glob.glob(os.path.join(config_paths.EPH_CHUNKS_DIR, "*.npz")))
    index = []
    for path in all_chunks:
        try:
            data = np.load(path, allow_pickle=True)
            sid = str(data["subject_id"])
            if sid in test_subjects:
                scen = config_paths.scenario_to_index(str(data["scenario"]))
                index.append({"chunk": os.path.basename(path), "path": path,
                              "subject_id": sid, "scenario": scen})
        except Exception as e:
            print(f"[warn] could not read {path}: {e}")
    return index

TEST_INDEX = build_test_index()
print(f"[note] test segments found: {len(TEST_INDEX)} (expected 453)")
if TEST_INDEX:
    _sc = pd.Series([r["subject_id"] for r in TEST_INDEX]).value_counts()
    print("[note] segments per subject:\n" + _sc.to_string())

[note] test subjects from split: ['S009', 'S019', 'S023']
[note] test segments found: 453 (expected 453)
[note] segments per subject:
S023    225
S019    225
S009      3


In [12]:
# ---------------------------------------------------------------------------
# Cell 5 - Per-segment metrics for every model (resumable) -> tidy CSV
# ---------------------------------------------------------------------------
PERSEG_CSV = os.path.join(OUT, "exp2_persegment_metrics.csv")

def preprocess_pair(ecg_raw, pcg_raw):
    """Paper pipeline: polarity-fix ECG, then z-score ECG and PCG."""
    ecg = MET.normalize_zscore(MET.ensure_positive_polarity(ecg_raw.astype(np.float32)))
    pcg = MET.normalize_zscore(pcg_raw.astype(np.float32))
    return ecg, pcg

def segment_metrics(real_z, fake):
    """All five fidelity metrics on the z-scored ECG vs the generator output."""
    out = {
        "psnr": MET.calc_psnr(real_z, fake),
        "corr": MET.calc_correlation(real_z, fake),
        "rmse": MET.calc_rmse(real_z, fake),
        "rpeak_amp_err": MET.calc_r_peak_amplitude_error(
            real_z, fake, fs=config_paths.FS,
            distance=config_paths.RPEAK_DISTANCE,
            prominence=config_paths.RPEAK_PROMINENCE),
    }
    try:
        out["ssim"] = MET.calc_ssim(real_z, fake)
    except Exception:
        out["ssim"] = np.nan
    return out

def compute_persegment(index, models_dict):
    # Resume: reuse rows already cached for (chunk, model) pairs.
    done = set()
    cached_rows = []
    if os.path.exists(PERSEG_CSV) and not FORCE_RECOMPUTE:
        prev = pd.read_csv(PERSEG_CSV)
        cached_rows = prev.to_dict("records")
        done = {(r["chunk"], r["model"]) for r in cached_rows}
        print(f"[note] resuming: {len(done)} (chunk,model) rows already cached.")

    rows = list(cached_rows)
    n_new = 0
    for i, rec in enumerate(index):
        needed = [m for m in models_dict if (rec["chunk"], m) not in done]
        if not needed:
            continue
        try:
            data = np.load(rec["path"], allow_pickle=True)
            real_z, pcg_z = preprocess_pair(data["ecg"], data["pcg"])
        except Exception as e:
            print(f"[warn] skip {rec['chunk']}: {e}")
            continue

        pcg_t = torch.from_numpy(pcg_z).unsqueeze(0).unsqueeze(0).to(DEVICE)
        onehot = np.zeros(config_paths.NUM_SCENARIOS, dtype=np.float32)
        onehot[rec["scenario"]] = 1.0
        scen_t = torch.from_numpy(onehot).unsqueeze(0).to(DEVICE)

        for name in needed:
            is_prop = (name == "Proposed")
            try:
                fake = M.run_generator(models_dict[name], pcg_t, scen_t, is_prop)
                fake = np.asarray(fake).ravel()[:real_z.size]
                mvals = segment_metrics(real_z, fake)
            except Exception as e:
                print(f"[warn] inference failed {name} on {rec['chunk']}: {e}")
                mvals = {k: np.nan for k in METRIC_KEYS}
            rows.append({"chunk": rec["chunk"], "subject_id": rec["subject_id"],
                         "scenario": rec["scenario"],
                         "scenario_name": SCEN_NAME.get(rec["scenario"], "other"),
                         "model": name, **mvals})
            n_new += 1

        # Periodic checkpoint so a Colab disconnect never loses much work.
        if (i + 1) % 40 == 0:
            pd.DataFrame(rows).to_csv(PERSEG_CSV, index=False)
            print(f"  ...processed {i + 1}/{len(index)} segments "
                  f"({n_new} new rows) - checkpoint saved.")

    df = pd.DataFrame(rows)
    df.to_csv(PERSEG_CSV, index=False)
    print(f"[ok] per-segment metrics written: {PERSEG_CSV} "
          f"({len(df)} rows, {n_new} new)")
    return df

DF = compute_persegment(TEST_INDEX, {k: MODELS[k] for k in AVAILABLE})

# Keep only available models, in canonical order, and coerce numeric types.
DF = DF[DF["model"].isin(AVAILABLE)].copy()
for k in METRIC_KEYS:
    DF[k] = pd.to_numeric(DF[k], errors="coerce")

# Sanity check against the verified pooled Proposed numbers (17.10 dB, 0.610).
if "Proposed" in AVAILABLE:
    dp = DF[DF["model"] == "Proposed"]
    print(f"[check] Proposed pooled: PSNR={dp['psnr'].mean():.2f} dB, "
          f"corr={dp['corr'].mean():.3f}, "
          f"R-peak-amp-err valid N={dp['rpeak_amp_err'].notna().sum()}/{len(dp)}")

  ...processed 40/453 segments (40 new rows) - checkpoint saved.
  ...processed 80/453 segments (80 new rows) - checkpoint saved.
  ...processed 120/453 segments (120 new rows) - checkpoint saved.
  ...processed 160/453 segments (160 new rows) - checkpoint saved.
  ...processed 200/453 segments (200 new rows) - checkpoint saved.
  ...processed 240/453 segments (240 new rows) - checkpoint saved.
  ...processed 280/453 segments (280 new rows) - checkpoint saved.
  ...processed 320/453 segments (320 new rows) - checkpoint saved.
  ...processed 360/453 segments (360 new rows) - checkpoint saved.
  ...processed 400/453 segments (400 new rows) - checkpoint saved.
  ...processed 440/453 segments (440 new rows) - checkpoint saved.
[ok] per-segment metrics written: /content/drive/MyDrive/JBHI_Revision/exp2_baseline_stats/exp2_persegment_metrics.csv (453 rows, 453 new)
[check] Proposed pooled: PSNR=17.10 dB, corr=0.610, R-peak-amp-err valid N=453/453


In [13]:
# ---------------------------------------------------------------------------
# Cell 6 - Paired Wilcoxon signed-rank + Holm-Bonferroni
# ---------------------------------------------------------------------------
from scipy.stats import wilcoxon

def holm_bonferroni(pvals):
    """Holm-Bonferroni step-down adjusted p-values (order preserved)."""
    p = np.asarray(pvals, dtype=float)
    m = p.size
    order = np.argsort(p)
    adj = np.empty(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        val = (m - rank) * p[idx]
        running = max(running, val)          # enforce monotonicity
        adj[idx] = min(running, 1.0)
    return adj

def star(p):
    if not np.isfinite(p):
        return ""
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 5e-2:
        return "*"
    return ""

# Wide pivot per metric: index=chunk, columns=model. This pairs by segment.
def wide(metric):
    return DF.pivot_table(index="chunk", columns="model", values=metric)

paired = []   # one entry per (metric, baseline) comparison
if "Proposed" in AVAILABLE:
    for mkey, mlabel, direction, _ in METRICS:
        W = wide(mkey)
        if "Proposed" not in W.columns:
            continue
        for base in [b for b in BASELINES if b in W.columns]:
            sub = W[["Proposed", base]].dropna()
            n = len(sub)
            stat = pval = np.nan
            med_diff = np.nan
            prop_better = None
            if n >= 1:
                dvec = (sub["Proposed"] - sub[base]).values
                med_diff = float(np.median(dvec))
                if direction == "max":
                    prop_better = med_diff > 0
                else:
                    prop_better = med_diff < 0
                if np.any(dvec != 0):
                    try:
                        stat, pval = wilcoxon(sub["Proposed"].values, sub[base].values,
                                              zero_method="wilcox", alternative="two-sided")
                        stat, pval = float(stat), float(pval)
                    except Exception as e:
                        print(f"[warn] wilcoxon {mkey} vs {base}: {e}")
            paired.append({"metric": mkey, "metric_label": mlabel, "baseline": base,
                           "n_pairs": int(n), "statistic": stat, "p_raw": pval,
                           "median_diff_proposed_minus_base": med_diff,
                           "proposed_better": prop_better})

    # Holm-Bonferroni across the whole family of valid comparisons.
    valid = [i for i, r in enumerate(paired) if np.isfinite(r["p_raw"])]
    if valid:
        adj = holm_bonferroni([paired[i]["p_raw"] for i in valid])
        for j, i in enumerate(valid):
            paired[i]["p_holm"] = float(adj[j])
    for r in paired:
        r.setdefault("p_holm", np.nan)
        # Marker: stars if Proposed significantly better; dagger if significantly worse.
        sig = np.isfinite(r["p_holm"]) and r["p_holm"] < 0.05
        if sig and r["proposed_better"]:
            r["marker"] = star(r["p_holm"])
        elif sig and r["proposed_better"] is False:
            r["marker"] = "\\dagger"
        else:
            r["marker"] = ""

PAIRED_DF = pd.DataFrame(paired)
PAIRED_CSV = os.path.join(OUT, "exp2_paired_wilcoxon.csv")
PAIRED_DF.to_csv(PAIRED_CSV, index=False)
print(f"[ok] paired tests written: {PAIRED_CSV}")
if not PAIRED_DF.empty:
    print(PAIRED_DF[["metric", "baseline", "n_pairs", "statistic",
                     "p_raw", "p_holm", "marker"]].to_string(index=False))

[ok] paired tests written: /content/drive/MyDrive/JBHI_Revision/exp2_baseline_stats/exp2_paired_wilcoxon.csv


In [14]:
# ---------------------------------------------------------------------------
# Cell 7 - Bootstrap 95% CIs for the Proposed correlation
# ---------------------------------------------------------------------------
# Subject-clustered (block) bootstrap resamples the THREE recordings with
# replacement and recomputes the pooled segment-mean correlation. This honestly
# reflects the effective N ~ 3 and is far wider than the naive per-segment CI.
def bootstrap_cis():
    if "Proposed" not in AVAILABLE:
        return {}
    dp = DF[DF["model"] == "Proposed"].dropna(subset=["corr"])
    vals = dp["corr"].values
    point = float(np.mean(vals))

    # Naive segment-level bootstrap (treats 453 segments as independent).
    naive = np.array([np.mean(RNG.choice(vals, size=vals.size, replace=True))
                      for _ in range(N_BOOT)])
    naive_ci = [float(np.percentile(naive, 2.5)), float(np.percentile(naive, 97.5))]

    # Naive normal-approx CI for reference.
    sem = float(np.std(vals, ddof=1) / np.sqrt(vals.size))
    normal_ci = [point - 1.96 * sem, point + 1.96 * sem]

    # Subject-clustered bootstrap.
    subjects = sorted(dp["subject_id"].unique())
    subj_vals = {s: dp[dp["subject_id"] == s]["corr"].values for s in subjects}
    cluster_means = []
    for _ in range(N_BOOT):
        chosen = RNG.choice(subjects, size=len(subjects), replace=True)
        pooled = np.concatenate([subj_vals[s] for s in chosen])
        cluster_means.append(np.mean(pooled))
    cluster_means = np.array(cluster_means)
    cluster_ci = [float(np.percentile(cluster_means, 2.5)),
                  float(np.percentile(cluster_means, 97.5))]

    per_subject = {s: {"n": int(subj_vals[s].size),
                       "corr_mean": float(np.mean(subj_vals[s]))} for s in subjects}

    out = {
        "point_estimate_pooled_corr": point,
        "n_segments": int(vals.size),
        "n_subjects": len(subjects),
        "naive_segment_bootstrap_ci95": naive_ci,
        "naive_normal_approx_ci95": normal_ci,
        "subject_clustered_bootstrap_ci95": cluster_ci,
        "n_boot": N_BOOT,
        "per_subject": per_subject,
    }
    print(f"[boot] Proposed pooled corr = {point:.3f}")
    print(f"[boot] naive segment CI95      = [{naive_ci[0]:.3f}, {naive_ci[1]:.3f}]")
    print(f"[boot] subject-clustered CI95  = [{cluster_ci[0]:.3f}, {cluster_ci[1]:.3f}]  "
          f"(much wider: effective N ~ {len(subjects)})")
    return out

BOOT = bootstrap_cis()

[boot] Proposed pooled corr = 0.610
[boot] naive segment CI95      = [0.572, 0.646]
[boot] subject-clustered CI95  = [0.314, 0.907]  (much wider: effective N ~ 3)


In [15]:
# ---------------------------------------------------------------------------
# Cell 8 - Per-subject and per-scenario summary tables
# ---------------------------------------------------------------------------
def summarize(group_cols):
    recs = []
    for keys, g in DF.groupby(group_cols + ["model"]):
        keys = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(group_cols + ["model"], keys))
        row["n_segments"] = int(g.shape[0])
        for k in METRIC_KEYS:
            v = g[k].dropna()
            row[f"{k}_mean"] = float(v.mean()) if len(v) else np.nan
            row[f"{k}_std"] = float(v.std(ddof=1)) if len(v) > 1 else np.nan
            row[f"{k}_validN"] = int(len(v))
        recs.append(row)
    out = pd.DataFrame(recs)
    # Stable ordering by model.
    out["model"] = pd.Categorical(out["model"], categories=MODEL_ORDER, ordered=True)
    return out.sort_values(group_cols + ["model"]).reset_index(drop=True)

OVERALL = summarize([])            # per-model over all segments
BY_SUBJECT = summarize(["subject_id"])
BY_SCENARIO = summarize(["scenario_name"])

OVERALL.to_csv(os.path.join(OUT, "exp2_summary_overall.csv"), index=False)
BY_SUBJECT.to_csv(os.path.join(OUT, "exp2_summary_by_subject.csv"), index=False)
BY_SCENARIO.to_csv(os.path.join(OUT, "exp2_summary_by_scenario.csv"), index=False)
print("[ok] summary tables written (overall / by-subject / by-scenario).")

# One machine-readable JSON with everything a LaTeX snippet might cite.
def _json_default(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return None if np.isnan(o) else float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)

RESULTS_JSON = os.path.join(OUT, "exp2_results.json")
with open(RESULTS_JSON, "w") as f:
    json.dump({
        "n_test_segments": len(TEST_INDEX),
        "models_available": AVAILABLE,
        "metrics": METRIC_KEYS,
        "overall": OVERALL.to_dict("records"),
        "by_subject": BY_SUBJECT.to_dict("records"),
        "by_scenario": BY_SCENARIO.to_dict("records"),
        "paired_wilcoxon": PAIRED_DF.to_dict("records") if not PAIRED_DF.empty else [],
        "bootstrap_correlation": BOOT,
    }, f, indent=2, default=_json_default)
print(f"[ok] results JSON written: {RESULTS_JSON}")

[ok] summary tables written (overall / by-subject / by-scenario).
[ok] results JSON written: /content/drive/MyDrive/JBHI_Revision/exp2_baseline_stats/exp2_results.json


In [16]:
# ---------------------------------------------------------------------------
# Cell 9 - Paste-ready LaTeX tables
# ---------------------------------------------------------------------------
# Marker lookup: (metric, baseline) -> significance marker vs Proposed.
MARK = {}
if not PAIRED_DF.empty:
    for _, r in PAIRED_DF.iterrows():
        MARK[(r["metric"], r["baseline"])] = r["marker"]

def best_model_for(mkey, direction):
    means = {m: OVERALL.loc[OVERALL["model"] == m, f"{mkey}_mean"].values for m in AVAILABLE}
    means = {m: v[0] for m, v in means.items() if len(v) and np.isfinite(v[0])}
    if not means:
        return None
    return (max if direction == "max" else min)(means, key=means.get)

def write_main_table():
    lines = []
    lines.append("% ============================================================")
    lines.append("% exp2 main summary table - Reviewer 1, comment 2")
    lines.append("% Per-segment metrics over all 453 EPHNOGRAM test segments.")
    lines.append("% Markers: significance of Proposed vs that baseline,")
    lines.append("%   paired Wilcoxon signed-rank, Holm-Bonferroni corrected.")
    lines.append("%   * p<0.05, ** p<0.01, *** p<0.001; dagger = Proposed worse.")
    lines.append("% FIND the current Table II and REPLACE it with this block "
                 "(update the label if needed).")
    lines.append("% ============================================================")
    lines.append("\\begin{table}[t]")
    lines.append("\\centering")
    lines.append("\\caption{Per-segment reconstruction quality on the EPHNOGRAM "
                 "test set ($N=453$ segments, three recordings). Values are "
                 "mean\\,$\\pm$\\,standard deviation. Significance markers denote "
                 "the paired Wilcoxon signed-rank comparison of the Proposed "
                 "model against each baseline, after Holm-Bonferroni correction "
                 "($^{*}p<0.05$, $^{**}p<0.01$, $^{***}p<0.001$). Best value per "
                 "column is in bold.}")
    lines.append("\\label{tab:persegment_stats}")
    ncols = len(METRICS)
    lines.append("\\setlength{\\tabcolsep}{4pt}")
    lines.append("\\begin{tabular}{l" + "c" * ncols + "}")
    lines.append("\\toprule")
    header = "Model & " + " & ".join(m[1] for m in METRICS) + " \\\\"
    lines.append(header)
    lines.append("\\midrule")
    best = {mkey: best_model_for(mkey, d) for mkey, _, d, _ in METRICS}
    for model in AVAILABLE:
        cells = [MODEL_PRETTY[model]]
        for mkey, _, _, prec in METRICS:
            row = OVERALL.loc[OVERALL["model"] == model]
            mean = row[f"{mkey}_mean"].values[0] if len(row) else np.nan
            std = row[f"{mkey}_std"].values[0] if len(row) else np.nan
            if not np.isfinite(mean):
                cells.append("--")
                continue
            txt = f"{mean:.{prec}f}$\\pm${std:.{prec}f}" if np.isfinite(std) \
                  else f"{mean:.{prec}f}"
            marker = MARK.get((mkey, model), "")
            if marker:
                txt = txt + f"$^{{{marker}}}$"
            if best.get(mkey) == model:
                txt = f"\\textbf{{{txt}}}"
            cells.append(txt)
        lines.append(" & ".join(cells) + " \\\\")
    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    lines.append("\\end{table}")
    path = os.path.join(OUT, "exp2_summary_table.tex")
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")
    print(f"[ok] LaTeX main table: {path}")
    return path

def write_persubject_table():
    """Per-subject correlation for each model - the R1.2 headline evidence."""
    lines = []
    lines.append("% ============================================================")
    lines.append("% exp2 per-subject correlation table - Reviewer 1, comment 2")
    lines.append("% Shows the headline pooled correlation is one strong subject")
    lines.append("% (S023) averaged with one weak subject (S019); S009 has n=3.")
    lines.append("% ============================================================")
    subjects = list(BY_SUBJECT["subject_id"].unique())
    lines.append("\\begin{table}[t]")
    lines.append("\\centering")
    lines.append("\\caption{Per-recording Pearson correlation $r$ "
                 "(mean\\,$\\pm$\\,standard deviation) on the EPHNOGRAM test set. "
                 "The pooled correlation is dominated by subject variance: a "
                 "strong exercise recording (S023) is averaged with a weak one "
                 "(S019). The rest recording (S009) contributes only three "
                 "segments.}")
    lines.append("\\label{tab:persubject_corr}")
    lines.append("\\setlength{\\tabcolsep}{5pt}")
    lines.append("\\begin{tabular}{l" + "c" * len(subjects) + "c}")
    lines.append("\\toprule")
    subj_hdr = " & ".join(f"{s}" for s in subjects)
    lines.append(f"Model & {subj_hdr} & Pooled \\\\")
    lines.append("\\midrule")
    for model in AVAILABLE:
        cells = [MODEL_PRETTY[model]]
        for s in subjects:
            row = BY_SUBJECT[(BY_SUBJECT["subject_id"] == s) & (BY_SUBJECT["model"] == model)]
            if len(row):
                mean = row["corr_mean"].values[0]
                std = row["corr_std"].values[0]
                n = int(row["n_segments"].values[0])
                cell = f"{mean:.3f}$\\pm${std:.3f}" if np.isfinite(std) else f"{mean:.3f}"
                cell = cell + f" ($n{{=}}{n}$)"
            else:
                cell = "--"
            cells.append(cell)
        prow = OVERALL[OVERALL["model"] == model]
        pooled = prow["corr_mean"].values[0] if len(prow) else np.nan
        cells.append(f"{pooled:.3f}" if np.isfinite(pooled) else "--")
        lines.append(" & ".join(cells) + " \\\\")
    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    lines.append("\\end{table}")
    path = os.path.join(OUT, "exp2_persubject_corr_table.tex")
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")
    print(f"[ok] LaTeX per-subject table: {path}")
    return path

def write_bootstrap_note():
    """A one-paragraph LaTeX note stating both CIs for the response letter."""
    if not BOOT:
        return None
    p = BOOT["point_estimate_pooled_corr"]
    nlo, nhi = BOOT["naive_segment_bootstrap_ci95"]
    clo, chi = BOOT["subject_clustered_bootstrap_ci95"]
    txt = (
        "% exp2 bootstrap CI note - drop into the R1.2 response.\n"
        f"The Proposed model reaches a pooled Pearson correlation of "
        f"{p:.3f} on the EPHNOGRAM test set. A naive segment-level bootstrap "
        f"gives a narrow 95\\% confidence interval of "
        f"$[{nlo:.3f},\\,{nhi:.3f}]$. That interval is optimistic, because the "
        f"453 segments come from only three continuous recordings. A "
        f"subject-clustered bootstrap that resamples the three recordings "
        f"widens the interval to $[{clo:.3f},\\,{chi:.3f}]$, which honestly "
        f"reflects the effective sample size.\n")
    path = os.path.join(OUT, "exp2_bootstrap_note.tex")
    with open(path, "w") as f:
        f.write(txt)
    print(f"[ok] LaTeX bootstrap note: {path}")
    return path

TAB_MAIN = write_main_table()
TAB_SUBJ = write_persubject_table()
NOTE_BOOT = write_bootstrap_note()

[ok] LaTeX main table: /content/drive/MyDrive/JBHI_Revision/exp2_baseline_stats/exp2_summary_table.tex
[ok] LaTeX per-subject table: /content/drive/MyDrive/JBHI_Revision/exp2_baseline_stats/exp2_persubject_corr_table.tex
[ok] LaTeX bootstrap note: /content/drive/MyDrive/JBHI_Revision/exp2_baseline_stats/exp2_bootstrap_note.tex


In [17]:
# ---------------------------------------------------------------------------
# Cell 10 - IEEE FIGURE A (double column): correlation box plots by subject
# ---------------------------------------------------------------------------
# Box plot of per-segment Pearson correlation for each model, faceted by
# recording plus a pooled panel. Boxes coloured by model; one figure-level
# legend (no overlap); bold axis label with units.
import matplotlib.patches as mpatches

def figure_boxplot_by_subject():
    IE.set_ieee_style()
    subjects = list(pd.Series([r["subject_id"] for r in TEST_INDEX]).unique())
    subjects = sorted(subjects)
    panels = subjects + ["Pooled"]
    fig, axes = IE.new_fig("double", height=2.7, nrows=1, ncols=len(panels),
                           sharey=True)
    if len(panels) == 1:
        axes = [axes]

    for ax, panel in zip(axes, panels):
        if panel == "Pooled":
            sub = DF
            n_seg = DF["chunk"].nunique()
            title = f"Pooled ($n{{=}}{n_seg}$)"
        else:
            sub = DF[DF["subject_id"] == panel]
            n_seg = sub["chunk"].nunique()
            title = f"{panel} ($n{{=}}{n_seg}$)"

        data, colors, positions = [], [], []
        for i, model in enumerate(AVAILABLE):
            vals = sub[sub["model"] == model]["corr"].dropna().values
            data.append(vals if vals.size else [np.nan])
            colors.append(MODEL_COLOR[model])
            positions.append(i + 1)

        bp = ax.boxplot(data, positions=positions, widths=0.62,
                        patch_artist=True, showfliers=False,
                        medianprops=dict(color="black", linewidth=1.1),
                        whiskerprops=dict(linewidth=0.9),
                        capprops=dict(linewidth=0.9),
                        boxprops=dict(linewidth=0.8))
        for patch, c in zip(bp["boxes"], colors):
            patch.set_facecolor(c)
            patch.set_alpha(0.85)
        # Overlay jittered points for the tiny-n panels so single boxes read.
        for i, model in enumerate(AVAILABLE):
            vals = sub[sub["model"] == model]["corr"].dropna().values
            if 0 < vals.size <= 12:
                jit = (RNG.random(vals.size) - 0.5) * 0.28
                ax.scatter(np.full(vals.size, i + 1) + jit, vals, s=6,
                           color="black", alpha=0.5, zorder=3, linewidths=0)

        ax.set_xticks(range(1, len(AVAILABLE) + 1))
        ax.set_xticklabels([])                       # models carried by legend
        ax.set_title(title)
        ax.axhline(0.0, color=IE.COLORS["muted"], linewidth=0.6, linestyle="--")
        IE.finalize_axes(ax)

    axes[0].set_ylabel("Pearson correlation $r$", fontweight="bold")

    handles = [mpatches.Patch(facecolor=MODEL_COLOR[m], edgecolor="0.3",
                              label=MODEL_PRETTY[m]) for m in AVAILABLE]
    try:
        fig.legend(handles=handles, loc="outside lower center",
                   ncol=len(AVAILABLE), frameon=True)
    except Exception:
        fig.legend(handles=handles, loc="lower center", ncol=len(AVAILABLE),
                   bbox_to_anchor=(0.5, -0.04), frameon=True)
    IE.save_fig(fig, OUT, "exp2_fig_corr_boxplot_by_subject")

if len(DF):
    figure_boxplot_by_subject()

  saved  /content/drive/MyDrive/JBHI_Revision/exp2_baseline_stats/exp2_fig_corr_boxplot_by_subject.pdf
  saved  /content/drive/MyDrive/JBHI_Revision/exp2_baseline_stats/exp2_fig_corr_boxplot_by_subject.png


In [18]:
# ---------------------------------------------------------------------------
# Cell 11 - IEEE FIGURE B (double column): grouped bars across metrics
# ---------------------------------------------------------------------------
# One panel per metric; four coloured bars (models) with std error bars.
# Bold per-panel axis label carrying units; shared figure legend, no overlap.
def figure_grouped_bars():
    IE.set_ieee_style()
    fig, axes = IE.new_fig("double", height=2.6, nrows=1, ncols=len(METRICS))
    if len(METRICS) == 1:
        axes = [axes]
    for ax, (mkey, mlabel, direction, prec) in zip(axes, METRICS):
        means, stds, colors = [], [], []
        for model in AVAILABLE:
            row = OVERALL[OVERALL["model"] == model]
            means.append(row[f"{mkey}_mean"].values[0] if len(row) else np.nan)
            stds.append(row[f"{mkey}_std"].values[0] if len(row) else np.nan)
            colors.append(MODEL_COLOR[model])
        x = np.arange(len(AVAILABLE))
        ax.bar(x, means, yerr=stds, color=colors, alpha=0.9,
               edgecolor="0.25", linewidth=0.6,
               error_kw=dict(elinewidth=0.8, capsize=2, capthick=0.8, ecolor="0.3"))
        ax.set_xticks(x)
        ax.set_xticklabels([])
        ax.set_ylabel(mlabel, fontweight="bold")
        arrow = r"$\uparrow$" if direction == "max" else r"$\downarrow$"
        ax.set_title(arrow, fontweight="bold")
        IE.finalize_axes(ax)

    handles = [mpatches.Patch(facecolor=MODEL_COLOR[m], edgecolor="0.3",
                              label=MODEL_PRETTY[m]) for m in AVAILABLE]
    try:
        fig.legend(handles=handles, loc="outside lower center",
                   ncol=len(AVAILABLE), frameon=True)
    except Exception:
        fig.legend(handles=handles, loc="lower center", ncol=len(AVAILABLE),
                   bbox_to_anchor=(0.5, -0.04), frameon=True)
    IE.save_fig(fig, OUT, "exp2_fig_metric_bars")

if len(OVERALL):
    figure_grouped_bars()

  saved  /content/drive/MyDrive/JBHI_Revision/exp2_baseline_stats/exp2_fig_metric_bars.pdf
  saved  /content/drive/MyDrive/JBHI_Revision/exp2_baseline_stats/exp2_fig_metric_bars.png


In [19]:
# ---------------------------------------------------------------------------
# Cell 12 - Manifest
# ---------------------------------------------------------------------------
print("\n" + "=" * 64)
print("exp2 COMPLETE - artifacts in:", OUT)
for fn in sorted(os.listdir(OUT)):
    print("   ", fn)
print("=" * 64)


exp2 COMPLETE - artifacts in: /content/drive/MyDrive/JBHI_Revision/exp2_baseline_stats
    exp2_bootstrap_note.tex
    exp2_fig_corr_boxplot_by_subject.pdf
    exp2_fig_corr_boxplot_by_subject.png
    exp2_fig_metric_bars.pdf
    exp2_fig_metric_bars.png
    exp2_paired_wilcoxon.csv
    exp2_persegment_metrics.csv
    exp2_persubject_corr_table.tex
    exp2_results.json
    exp2_summary_by_scenario.csv
    exp2_summary_by_subject.csv
    exp2_summary_overall.csv
    exp2_summary_table.tex
